In [1]:
import os
import random
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.sparse import save_npz
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
movies = pd.read_csv(r"C:\Users\ACER\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Python 3.12\movies_cleaned.csv")
ratings = pd.read_csv(r"C:\Users\ACER\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Python 3.12\ratings_cleaned.csv")

In [3]:

movies['genres'] = movies['genres'].fillna('').str.replace('|', ' ')
movies['content'] = movies['genres']

In [4]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [5]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


In [6]:
os.makedirs("models", exist_ok=True)
save_npz("models/tfidf_matrix.npz", tfidf_matrix)
np.save("models/cosine_sim.npy", cosine_sim)
joblib.dump(tfidf, "models/tfidf_vectorizer.joblib")
movies.to_csv("models/movies_with_content.csv", index=False)

In [7]:
def recommend_movies_for_user(user_id, ratings, movies, indices, cosine_sim, top_n=10):
    liked_movies = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4)]
    
    if liked_movies.empty:
        print(f"User {user_id} không có đánh giá >= 4 để làm gợi ý.")
        return pd.DataFrame()

    agg_scores = {}
    contribution_count = {}

    for movie_id in liked_movies['movieId'].values:
        movie_title = movies[movies['movieId'] == movie_id]['title'].values
        if not movie_title:
            continue
        movie_title = movie_title[0]
        
        if movie_title not in indices:
            continue
            
        idx = indices[movie_title]
        sim_scores = cosine_sim[idx].flatten()
        sim_scores = list(enumerate(sim_scores))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
        
        for i, score in sim_scores:
            if i in agg_scores:
                agg_scores[i] += score
                contribution_count[i] += 1
            else:
                agg_scores[i] = score
                contribution_count[i] = 1

    normalized_scores = []
    for i in agg_scores:
        avg_score = agg_scores[i] / contribution_count[i]
        normalized_scores.append((i, avg_score))
    
    sorted_scores = sorted(normalized_scores, key=lambda x: x[1], reverse=True)[:top_n]
    movie_indices = [i[0] for i in sorted_scores]
    scores = [i[1] for i in sorted_scores]
    
    result = movies.iloc[movie_indices][['movieId', 'title']].copy()
    result['similarity'] = scores
    
    return result


In [8]:
user_id = random.choice(ratings['userId'].unique())
liked_movies = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4)]

if not liked_movies.empty:
    print(f"\nUser {user_id} thích các phim sau:")
    for movie_id in liked_movies['movieId'].values:
        movie_title = movies[movies['movieId'] == movie_id]['title'].values[0]
        print(f"- {movie_title}")
    
    cbf_recs = recommend_movies_for_user(user_id, ratings, movies, indices, cosine_sim, top_n=10)
    
    if not cbf_recs.empty:
        print("\nTop phim gợi ý (CBF):")
        print(cbf_recs)
        
        # Visualize recommendations
        fig = px.bar(
            cbf_recs.sort_values("similarity"),
            x="similarity",
            y="title",
            orientation="h",
            title=f"Top 10 phim gợi ý cho user {user_id} dựa trên các phim đã thích",
            labels={"similarity": "Độ tương đồng (0-1)", "title": "Tên phim"},
            color="similarity",
            color_continuous_scale="viridis"
        )
        fig.update_layout(yaxis={'categoryorder': 'total ascending'})
        fig.show()
    else:
        print("Không thể tạo gợi ý do không tìm thấy phim phù hợp.")
else:
    print(f"User {user_id} không có đánh giá >= 4 để làm gợi ý.")


User 532 thích các phim sau:
- Heat (1995)
- Braveheart (1995)
- Taxi Driver (1976)
- Shawshank Redemption, The (1994)
- Fugitive, The (1993)
- Jurassic Park (1993)
- Rising Sun (1993)
- Schindler's List (1993)
- Terminator 2: Judgment Day (1991)
- Silence of the Lambs, The (1991)
- Fargo (1996)
- Mulholland Falls (1996)
- Godfather, The (1972)
- Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)
- Aliens (1986)
- Goodfellas (1990)
- Alien (1979)
- Boot, Das (Boat, The) (1981)
- Terminator, The (1984)
- Glory (1989)
- Miller's Crossing (1990)
- Indiana Jones and the Last Crusade (1989)
- Cape Fear (1991)
- Men in Black (a.k.a. MIB) (1997)
- Hunt for Red October, The (1990)
- Titanic (1997)
- French Connection, The (1971)
- Godfather: Part III, The (1990)
- Saving Private Ryan (1998)
- Untouchables, The (1987)
- Thing, The (1982)
- American History X (1998)
- King Kong (1933)
- Romancing the Stone (1984)
- Matrix, The (1999)
- American Beauty (1999)
- Fight 